# Test 2 — Prompt Length Sensitivity

## 1. Purpose

This test investigates how increasing **input prompt length** influences model behaviour and performance.

This focuses on **input complexity**, which directly affects:
- preprocessing time (tokenisation and prefill)
- memory usage
- attention computation cost

Understanding this is important for designing efficient prompts in HPC workflows.

### What to observe
- response relevance
- instruction adherence
- latency scaling
- stability with longer inputs

## 2. Setup

**Model:**
- Qwen/Qwen2.5-1.5B-Instruct

**Prompt structure:**
Four prompts of increasing length:
- short
- medium
- long
- very long

All prompts request a structured explanation of reproducibility in research.

**Parameters:**

| Parameter   | Value |
|---|---|
| temperature | 0.5 |
| max_tokens | 3000 |
| GPU | 1x GH200 |

## 3. Code and Output

In [ ]:
#!/bin/bash
#SBATCH --job-name=vllm-infer-t2
#SBATCH --partition=ghtest
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --gres=gpu:1
#SBATCH --time=00:25:00
#SBATCH -A <project_ID> 
#SBATCH --output=vllm_test2_%j.out
#SBATCH --error=vllm_test2_%j.err

set -euo pipefail

echo "===== JOB START ====="
echo "Host: $(hostname)"
echo "Date: $(date)"
echo "Using GPU:"
nvidia-smi

# -----------------------------
# User-editable section
# -----------------------------
PROJECT= <project-ID>
USER_NAME= <username>

BASE="/nobackup/projects/${PROJECT}/${USER_NAME}/modules/vllm26"
CONTAINER_DIR="/nobackup/projects/${PROJECT}/${USER_NAME}/containers"

IMAGE="${CONTAINER_DIR}/vllm-ag2-26.01.1.sif"

HOME_DIR="${BASE}/home"
WORK_DIR="${BASE}/workspace"
CACHE_DIR="${WORK_DIR}/.cache"
PY_FILE="${WORK_DIR}/test2_prompt_length.py"

# Fixed parameters for Test 2
MODEL="Qwen/Qwen2.5-1.5B-Instruct"
TEMPERATURE="0.5"
MAXTOKENS="3000"
# -----------------------------

if [[ ! -f "$IMAGE" ]]; then
    echo "ERROR: container image $IMAGE not found"
    exit 1
fi

mkdir -p "$HOME_DIR" "$WORK_DIR" "$CACHE_DIR"
chmod -R a+rwx "$BASE"

echo "===== WRITING PYTHON FILE ====="

cat > "$PY_FILE" <<'PYEOF'
from vllm import LLM, SamplingParams
import time
import torch
import os

MODEL_NAME = os.environ["TEST2_MODEL"]
TEMPERATURE = float(os.environ["TEST2_TEMPERATURE"])
MAX_TOKENS = int(os.environ["TEST2_MAXTOKENS"])

# -----------------------------
# Prompts of increasing length
# -----------------------------
prompts = [
    (
        "short",
        "Summarise the importance of reproducibility in scientific research."
    ),
    (
        "medium",
        """Summarise the importance of reproducibility in scientific research.
Include why reproducibility matters for trust, verification,
and long-term reuse of results. Keep the answer clear and suitable for a researcher new to the topic."""
    ),
    (
        "long",
        """Summarise the importance of reproducibility in scientific research.
Include why reproducibility matters for trust, verification,
and long-term reuse of results. Discuss how poor documentation,
missing dependencies, and inconsistent software environments can
make research harder to reproduce. Also explain how containerisation
and workflow documentation can help."""
    ),
    (
        "very_long",
        """Summarise the importance of reproducibility in scientific research.
Include why reproducibility matters for trust, verification,
and long-term reuse of results. Discuss how poor documentation,
missing dependencies, and inconsistent software environments can
make research harder to reproduce. Also explain how containerisation
and workflow documentation can help. Describe why reproducibility
is especially important in computational and AI-based research where
models, software libraries, GPU dependencies, and runtime environments
can all affect results."""
    )
]

print("===== TEST 2: PROMPT LENGTH REACTION =====")
print(f"Model: {MODEL_NAME}")
print(f"Temperature: {TEMPERATURE}")
print(f"Max tokens: {MAX_TOKENS}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("==========================================")

# -----------------------------
# Load model once
# -----------------------------
load_start = time.time()
llm = LLM(model=MODEL_NAME)
load_end = time.time()

print(f"\nModel load time: {load_end - load_start:.2f} seconds")

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS
)

results = []

# -----------------------------
# Run tests
# -----------------------------
for label, prompt in prompts:
    print("\n==================================================")
    print(f"Running prompt type: {label}")
    print("==================================================")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start_time = time.time()
    outputs = llm.generate([prompt], sampling_params)
    end_time = time.time()

    latency = end_time - start_time

    if torch.cuda.is_available():
        memory_used = torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        memory_used = -1.0

    prompt_length = len(prompt)
    response_text = outputs[0].outputs[0].text

    print(f"Prompt characters: {prompt_length}")
    print(f"Latency (s): {latency:.3f}")
    print(f"Peak GPU memory (MB): {memory_used:.2f}")
    print("\nResponse preview:")
    print(response_text[:300].replace("\n", " "))
    print("\n----- END PREVIEW -----")

    results.append({
        "prompt_type": label,
        "prompt_chars": prompt_length,
        "latency_seconds": latency,
        "gpu_memory_mb": memory_used,
    })

# -----------------------------
# Plain-text results table
# -----------------------------
print("\n\n========== TEST RESULTS ==========")
header = f"{'prompt_type':<12} {'prompt_chars':>12} {'latency_s':>12} {'gpu_mem_mb':>12}"
print(header)
print("-" * len(header))

for row in results:
    print(
        f"{row['prompt_type']:<12} "
        f"{row['prompt_chars']:>12} "
        f"{row['latency_seconds']:>12.3f} "
        f"{row['gpu_memory_mb']:>12.2f}"
    )
PYEOF

if [[ ! -s "$PY_FILE" ]]; then
    echo "ERROR: Python file was not created correctly: $PY_FILE"
    exit 1
fi

echo "Python file created:"
ls -l "$PY_FILE"
echo "----- preview -----"
head -n 20 "$PY_FILE"
echo "-------------------"

echo "===== RUNNING TEST 2 ====="
echo "Model: $MODEL"
echo "Temperature: $TEMPERATURE"
echo "Max tokens: $MAXTOKENS"

apptainer exec --nv \
  --bind "$WORK_DIR:/workspace" \
  --home "$HOME_DIR:/users/$USER_NAME" \
  --env HF_HOME=/workspace/.cache/huggingface \
  --env XDG_CACHE_HOME=/workspace/.cache \
  --env FLASHINFER_WORKSPACE_DIR=/users/$USER_NAME/.cache/flashinfer \
  --env TORCHINDUCTOR_CACHE_DIR=/workspace/.cache/torchinductor \
  --env TRITON_CACHE_DIR=/workspace/.cache/triton \
  --env VLLM_DISABLE_COMPILE=1 \
  --env TEST2_MODEL="$MODEL" \
  --env TEST2_TEMPERATURE="$TEMPERATURE" \
  --env TEST2_MAXTOKENS="$MAXTOKENS" \
  "$IMAGE" \
  python /workspace/test2_prompt_length.py

echo "===== JOB END ====="
date

In [ ]:
===== JOB START =====
Host: gh005.bede.dur.ac.uk
Date: Mon 16 Mar 18:11:25 GMT 2026
Using GPU:
Mon Mar 16 18:11:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GH200 480GB             On  |   00000009:01:00.0 Off |                    0 |
| N/A   33C    P0            107W /  900W |       6MiB /  97871MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|  No running processes found                                                             |
+-----------------------------------------------------------------------------------------+
===== WRITING PYTHON FILE =====
Python file created:
-rwxrwxrwx 1 <user_ID> <project_ID> 3979 Mar 16 18:11 /nobackup/projects/<project_ID>/<user_ID>/modules/vllm26/workspace/test2_prompt_length.py
----- preview -----
from vllm import LLM, SamplingParams
import time
import torch
import os

MODEL_NAME = os.environ["TEST2_MODEL"]
TEMPERATURE = float(os.environ["TEST2_TEMPERATURE"])
MAX_TOKENS = int(os.environ["TEST2_MAXTOKENS"])

# -----------------------------
# Prompts of increasing length
# -----------------------------
prompts = [
    (
        "short",
        "Summarise the importance of reproducibility in scientific research."
    ),
    (
        "medium",
        """Summarise the importance of reproducibility in scientific research.
-------------------
===== RUNNING TEST 2 =====
Model: Qwen/Qwen2.5-1.5B-Instruct
Temperature: 0.5
Max tokens: 3000
===== TEST 2: PROMPT LENGTH REACTION =====
Model: Qwen/Qwen2.5-1.5B-Instruct
Temperature: 0.5
Max tokens: 3000
CUDA available: True
==========================================
INFO 03-16 18:11:33 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
INFO 03-16 18:11:33 [model.py:514] Resolved architecture: Qwen2ForCausalLM
INFO 03-16 18:11:33 [model.py:1661] Using max model len 32768
INFO 03-16 18:11:34 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:34 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer=
'Qwen/Qwen2.5-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_forma
t=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_
config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False)
, observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=Fal
se, enable_layerwise_nvtx_tracing=False), seed=0, served_model_name=Qwen/Qwen2.5-1.5B-Instruct, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode
': <CompilationMode.VLLM_COMPILE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'v
llm::unified_attention_with_output', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2
_mamba_mixer', 'vllm::gdn_attention_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': 
{'enable_auto_functionalized_v2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 
'cudagraph_capture_sizes': [1, 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352,
 368, 384, 400, 416, 432, 448, 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_qua
nt': False, 'fuse_attn_quant': False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <Dyna
micShapesType.BACKED: 'backed'>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:36 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.20:44349 backend=nccl
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:36 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:36 [gpu_model_runner.py:3562] Starting to load model Qwen/Qwen2.5-1.5B-Instruct...
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:37 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:38 [weight_utils.py:527] No model.safetensors.index.json found in remote.
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:38 [default_loader.py:308] Loading weights took 0.39 seconds
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:38 [gpu_model_runner.py:3659] Model loading took 2.8871 GiB memory and 1.563258 seconds
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:40 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/9ae60163a559654a93f737cc67e21d734e3978c68d8cc40c5665
cd3c5231ef36/rank_0_0/model
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:42 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/8afbe179ad/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:42 [backends.py:703] Dynamo bytecode transform time: 3.23 s
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:44 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.767 s
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:44 [monitor.py:34] torch.compile takes 4.00 s in total
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:45 [gpu_worker.py:375] Available KV cache memory: 76.88 GiB
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:45 [kv_cache_utils.py:1291] GPU KV cache size: 2,879,232 tokens
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:45 [kv_cache_utils.py:1296] Maximum concurrency for 32,768 tokens per request: 87.87x
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:49 [gpu_model_runner.py:4587] Graph capturing finished in 4 secs, took -0.86 GiB
(EngineCore_DP0 pid=2940423) INFO 03-16 18:11:49 [core.py:259] init engine (profile, create kv cache, warmup model) took 10.93 seconds
INFO 03-16 18:11:50 [llm.py:360] Supported tasks: ['generate']

Model load time: 17.16 seconds

==================================================
Running prompt type: short
==================================================
Prompt characters: 67
Latency (s): 0.851
Peak GPU memory (MB): 0.00

Response preview:
 Reproducibility is a fundamental principle in scientific research that ensures that results can be replicated by other researchers under the same conditions. It is essential for several reasons:  1. **Trust in Re
search**: Reproducibility builds trust in the scientific community, allowing other rese

----- END PREVIEW -----

==================================================
Running prompt type: medium
==================================================
Prompt characters: 230
Latency (s): 1.172
Peak GPU memory (MB): 0.00

Response preview:
 Reproducibility is a cornerstone of scientific research that ensures the credibility and reliability of findings. It is essential for several reasons:  1. **Trust in Research**: When research is reproducible, it 
builds trust among scientists, policymakers, and the public. It demonstrates that the r

----- END PREVIEW -----

==================================================
Running prompt type: long
==================================================
Prompt characters: 363
Latency (s): 0.457
Peak GPU memory (MB): 0.00

Response preview:
 Finally, discuss the importance of open source and open data in fostering reproducibility. Reproducibility is crucial in scientific research for several reasons. It ensures that results can be verified by other r
esearchers, which is essential for the trustworthiness of scientific knowledge. Reprodu

----- END PREVIEW -----

==================================================
Running prompt type: very_long
==================================================
Prompt characters: 556
Latency (s): 8.030
Peak GPU memory (MB): 0.00

Response preview:
 Highlight the benefits of reproducibility for the scientific community, including increased transparency, reproducibility, and innovation. Finally, provide examples of how reproducibility has been achieved in dif
ferent scientific fields, such as genomics, climate science, and biophysics. Summarise 

----- END PREVIEW -----


========== TEST RESULTS ==========
prompt_type  prompt_chars    latency_s   gpu_mem_mb
---------------------------------------------------
short                  67        0.851         0.00
medium                230        1.172         0.00
long                  363        0.457         0.00
very_long             556        8.030         0.00
ERROR 03-16 18:12:00 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.
===== JOB END =====

## 4. Results

### Measured results

| prompt_type | chars | latency (s) |
|---|---|---|
| short | 67 | 0.851 |
| medium | 230 | 1.172 |
| long | 363 | 0.457 |
| very_long | 556 | 8.030 |

### Observed behaviour

- **Short prompt (67 chars)**
  - Fast response (~0.85s)
  - Clear and relevant output
  - Good instruction adherence

- **Medium prompt (230 chars)**
  - Slight increase in latency (~1.17s)
  - Improved structure and detail
  - Strong coherence

- **Long prompt (363 chars)**
  - Unexpected latency drop (~0.46s)
  - Output begins mid-instruction
  - Signs of instruction misalignment

- **Very long prompt (556 chars)**
  - Significant latency increase (~8.03s)
  - Output shows prompt leakage and repetition
  - Reduced clarity and focus

### System behaviour

- Model load time: ~17 seconds
- Large spike in latency for very long prompt

## 5. Analysis

### 1 - Prompt length is not linear

Latency does not scale smoothly with input size:
- small increases → minimal cost
- larger prompts → sudden spikes

This reflects **attention complexity and batching behaviour**.

### 2 - Instruction dilution

As prompt length increases:
- instructions become less dominant
- model may focus on later parts of prompt
- earlier intent is partially ignored

This explains:
- mid-sentence outputs
- reduced coherence

### 3 - Prompt leakage

The very long prompt shows:
- repetition of prompt phrases
- transformation instead of answering

This indicates the model is:
- struggling to separate instruction from content
- approaching its effective reasoning limit

### 4 - HPC implications

Long prompts:
- increase prefill cost significantly
- reduce throughput
- can destabilise runs

Efficient prompt design is therefore critical:
- concise but structured prompts perform best
- excessive context can reduce quality


### Summary

This test highlights an important principle:

> Better prompts are not longer prompts — they are clearer prompts.

In HPC environments, prompt efficiency directly affects:
- runtime performance
- system stability
- research reproducibility